# Spam Email Classification (Improved Version)
Fixes applied: no data leakage in vectorization, stratified split, no `inplace=True`, full evaluation metrics, TF-IDF comparison.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle

## 1. Load and Clean Data

In [ ]:
data = pd.read_csv("spam.csv", encoding="latin-1")
data.head()

In [ ]:
# FIX: avoid inplace=True, reassign instead (safer pandas pattern)
data = data.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1)
data['class'] = data['class'].map({'ham': 0, 'spam': 1})
data.head()

In [ ]:
data.isnull().sum()

## 2. Train-Test Split (BEFORE vectorizing, with stratify)
FIX: We split the raw text first, then fit the vectorizer only on training data.
This avoids data leakage (test vocabulary influencing the model) and `stratify=y` preserves the 86.6/13.4 class ratio in both sets.

In [ ]:
X_raw = data['message']
y = data['class']

x_train_raw, x_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print("Train class ratio:", y_train.value_counts(normalize=True).to_dict())
print("Test class ratio:", y_test.value_counts(normalize=True).to_dict())

## 3. Vectorization + Model Training + Full Evaluation
Wrapped in a function so we can fairly compare CountVectorizer vs TfidfVectorizer with identical splits.

In [ ]:
def train_and_evaluate(vectorizer, name):
    # FIX: fit_transform ONLY on training text, transform (not fit) on test text
    x_train = vectorizer.fit_transform(x_train_raw)
    x_test = vectorizer.transform(x_test_raw)

    model = MultinomialNB()
    model.fit(x_train, y_train)

    preds = model.predict(x_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)

    print(f"--- {name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("Confusion Matrix [[TN, FP], [FN, TP]]:")
    print(cm)
    print()

    return model, vectorizer, {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

In [ ]:
count_model, count_vectorizer, count_metrics = train_and_evaluate(CountVectorizer(), "CountVectorizer + MultinomialNB")

In [ ]:
tfidf_model, tfidf_vectorizer, tfidf_metrics = train_and_evaluate(TfidfVectorizer(), "TfidfVectorizer + MultinomialNB")

## 4. Comparison Summary
We choose **CountVectorizer** as the final model since it has the higher F1-score (better precision/recall balance) on this dataset.
TF-IDF gives perfect precision but noticeably lower recall — a real precision/recall tradeoff worth being able to explain.

In [ ]:
comparison = pd.DataFrame({
    "CountVectorizer": count_metrics,
    "TF-IDF": tfidf_metrics
}).T
comparison

## 5. Manual Sanity Check (final chosen model: CountVectorizer)

In [ ]:
test_messages = ["You Won 500$", "Hey, are we still meeting for lunch tomorrow?"]
vect = count_vectorizer.transform(test_messages)
preds = count_model.predict(vect)
probs = count_model.predict_proba(vect)

for msg, pred, prob in zip(test_messages, preds, probs):
    label = "SPAM" if pred == 1 else "HAM"
    print(f"'{msg}' -> {label}  (P(ham)={prob[0]:.4f}, P(spam)={prob[1]:.4f})")

## 6. Save Final Model + Vectorizer
FIX: consistent naming — saved directly as `vectorizer.pkl` (matches what `spamDetector.py` loads), no manual rename needed.

In [ ]:
pickle.dump(count_model, open('spam.pkl', 'wb'))
pickle.dump(count_vectorizer, open('vectorizer.pkl', 'wb'))

print("Saved spam.pkl and vectorizer.pkl successfully.")

## Happy Coding